In [34]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

In [3]:
df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")

df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [4]:
df["Partner"] = df["Partner"].fillna(df["Partner"].mode()[0])
df["MonthlyCharges"] = df["MonthlyCharges"].fillna(df["MonthlyCharges"].median())
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df["TotalCharges"] = df["TotalCharges"].fillna(df["TotalCharges"].median())

Step 1: Create at least two new engineered features for a provided dataset and justify each in Markdown.

In [5]:
df["tenure"] = df["tenure"].replace({ 0 : 1})
df['avg_monthly_charge'] = df['TotalCharges'] / df['tenure']

df['avg_monthly_charge'].head()

0    29.850000
1    55.573529
2    54.075000
3    40.905556
4    75.825000
Name: avg_monthly_charge, dtype: float64

this shows the customer's avg monthly payment ower their whole time with the company. it is more stable than total charges, which just growswith tenure and doesn't tell us much on its own.

In [6]:
df['tenure_group'] = pd.cut(df["tenure"],
                            bins=[0,12,24,48,100],
                            labels=['new', 'established', 'loyal', 'veteran'])

df['tenure_group'].head()

0      new
1    loyal
2      new
3    loyal
4      new
Name: tenure_group, dtype: category
Categories (4, object): ['new' < 'established' < 'loyal' < 'veteran']

I grouped tenure into 4 bins: new (0-12 months), establushed (12-24), loyal (24-48), and veteran (48+). the relathionship between tenure and churn is probably not linear, new customers churn a lot more than long-term ones. binning helps the model capture this pattern more.

In [7]:
df["gender"] = df["gender"].replace({ "Male" : 1, "Female" : 0})
df["Partner"] = df["Partner"].replace({ "Yes" : 1, "No" : 0})
df["Dependents"] = df["Dependents"].replace({ "Yes" : 1, "No" : 0})
df["PhoneService"] = df["PhoneService"].replace({ "Yes" : 1, "No" : 0})
df["MultipleLines"] = df["MultipleLines"].replace({ "Yes" : 1, "No" : 0, "No phone service" : 0 })
df["OnlineSecurity"] = df["OnlineSecurity"].replace({ "Yes" : 1, "No" : 0, "No internet service" : 0 })
df["OnlineBackup"] = df["OnlineBackup"].replace({ "Yes" : 1, "No" : 0, "No internet service" : 0 })
df["DeviceProtection"] = df["DeviceProtection"].replace({ "Yes" : 1, "No" : 0, "No internet service" : 0 })
df["TechSupport"] = df["TechSupport"].replace({ "Yes" : 1, "No" : 0, "No internet service" : 0 })
df["StreamingTV"] = df["StreamingTV"].replace({ "Yes" : 1, "No" : 0, "No internet service" : 0 })
df["StreamingMovies"] = df["StreamingMovies"].replace({ "Yes" : 1, "No" : 0, "No internet service" : 0 })
df["PaperlessBilling"] = df["PaperlessBilling"].replace({ "Yes" : 1, "No" : 0 })
df["StreamingTV"] = df["StreamingTV"].replace({ "Yes" : 1, "No" : 0, "No internet service" : 0 })
df["Churn"] = df["Churn"].replace({"Yes": 1, "No": 0})
df = pd.get_dummies(df, columns=["InternetService", "Contract", "PaymentMethod", "tenure_group"])

C:\Users\darag\AppData\Local\Temp\ipykernel_8728\617814650.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["gender"] = df["gender"].replace({ "Male" : 1, "Female" : 0})
C:\Users\darag\AppData\Local\Temp\ipykernel_8728\617814650.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["Partner"] = df["Partner"].replace({ "Yes" : 1, "No" : 0})
C:\Users\darag\AppData\Local\Temp\ipykernel_8728\617814650.py:3: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old 

In [8]:
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,OnlineSecurity,OnlineBackup,...,Contract_One year,Contract_Two year,PaymentMethod_Bank transfer (automatic),PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,tenure_group_new,tenure_group_established,tenure_group_loyal,tenure_group_veteran
0,7590-VHVEG,0,0,1,0,1,0,0,0,1,...,False,False,False,False,True,False,True,False,False,False
1,5575-GNVDE,1,0,0,0,34,1,0,1,0,...,True,False,False,False,False,True,False,False,True,False
2,3668-QPYBK,1,0,0,0,2,1,0,1,1,...,False,False,False,False,False,True,True,False,False,False
3,7795-CFOCW,1,0,0,0,45,0,0,1,0,...,True,False,True,False,False,False,False,False,True,False
4,9237-HQITU,0,0,0,0,2,1,0,0,0,...,False,False,False,False,True,False,True,False,False,False


In [9]:
X = df.drop(columns=["customerID", "Churn"])
Y = df["Churn"]

In [10]:
X_temp, X_test, Y_temp, Y_test = train_test_split(
    X, Y, test_size=0.2, random_state=42)

In [12]:
X_train, X_val, Y_train, Y_val = train_test_split(
    X_temp, Y_temp, test_size=0.2, random_state=42)

In [36]:
scaler = StandardScaler()

X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index
)

X_val_scaled = pd.DataFrame(
    scaler.transform(X_val),
    columns=X_val.columns,
    index=X_val.index
)

X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),
    columns=X_test.columns,
    index=X_test.index
)

Step 2: Define a hyperparameter grid for a model from Week 3 (e.g. random forest)

In [19]:
param_grid = {
    "n_estimators": [200, 500],
    "max_depth": [10, 15],
    "class_weight": ["balanced", "balanced_subsample"]
}

Step 3: Run GridSearchCV with 5-fold cross-validation and report the best parameters and score.

In [21]:
grid = GridSearchCV(RandomForestClassifier(random_state=42),
                    param_grid, cv=5, scoring="f1", n_jobs=-1)
grid.fit(X_train, Y_train)
print("Best params :",grid.best_params_)
print("Best score :", grid.best_score_)
best_model = grid.best_estimator_

Best params : {'class_weight': 'balanced', 'max_depth': 10, 'n_estimators': 500}
Best score : 0.6207769966362157


Step 4: Compare the tuned model's cross-validated score against the untuned baseline from Week 3.

In [37]:
baseline_lr = LogisticRegression(class_weight='balanced', random_state=42)
baseline_lr.fit(X_train_scaled, Y_train)
baseline_f1 = f1_score(Y_train, baseline_lr.predict(X_train_scaled))

print(f"Grid Search CV score : {round(grid.best_score_, 4)}")
print(f"Week 3 baseline score : {round(baseline_f1, 4)}")

Grid Search CV score : 0.6208
Week 3 baseline score : 0.6235


Tuned Random Forest (GridSearchCV): F1 = 0.6208
Untuned Logistic Regression (Week 3 baseline): F1 = 0.6235

Even after tuning, Random Forest slightly underperforms the simpler 
Logistic Regression baseline. This suggests that for this churn dataset, 
a linear model with class balancing works better than a more complex 
ensemble model — feature engineering and class weighting matter more 
here than model complexity.

Step 5: Document which engineered feature and which hyperparameter mattered most.

tenure_group mattered the most. Splitting tenure into groups 
(new, established, loyal, veteran) helped the model see the non-linear 
relationship between tenure and churn — new customers churn much more 
than long-term ones.

`class_weight='balanced'` mattered the most. Without 
it, the F1 score for Random Forest was only 0.5674 (I check it). After adding it, 
the score improved to 0.6208. This makes sense because our churn data 
is imbalanced, so the model needs to pay more attention to the minority 
class (customers who churn).

`max_depth=10` and `n_estimators=500` also helped, but the effect was 
smaller compared to `class_weight`.